# Phase 17: YOLO11m + Copy-Paste Augmentation (FULL RUN)

This notebook trains the standard YOLO11m model on the augmented `dataset_yolo_single_class_cp` dataset, which artificially injects sub-0.5% area defects using Gaussian Alpha Blending. The goal is to boost Recall above the EXP-10 baseline (76.63%).

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install dependencies
!pip install ultralytics


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 1.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 10.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 8.9 MB/s eta 0:00:00


In [3]:
import os
import torch
from ultralytics import YOLO

PROJECT_ROOT = '/content/drive/MyDrive/sem_defect_project'
os.chdir(PROJECT_ROOT)
print(f"Current working directory: {os.getcwd()}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Current working directory: /content/drive/MyDrive/sem_defect_project
CUDA Available: True
GPU Device: Tesla T4


In [4]:
!unzip -o -q /content/drive/MyDrive/sem_defect_project/datasets/dataset_yolo_single_class_cp_fixed.zip -d /content/drive/MyDrive/sem_defect_project/datasets/


## 1. Load YOLO11m (Official Weights)

In [5]:
# Instantiate standard YOLO11m using official pretrained weights
print("Instantiating standard YOLO11m model...")
model = YOLO('yolo11m.pt')


Instantiating standard YOLO11m model...


In [7]:
# Ye code kharab path ko fix kar dega!
yaml_content = """path: /content/drive/MyDrive/sem_defect_project/datasets/dataset_yolo_single_class_cp
train: train/images
val: valid/images
test: test/images

nc: 1
names: ['Defect']
"""

with open('datasets/dataset_yolo_single_class_cp/data.yaml', 'w') as f:
    f.write(yaml_content)

print("data.yaml ka path successfully theek ho gaya hai!")


data.yaml ka path successfully theek ho gaya hai!


## 2. FULL Training Run (200 Epochs) on Augmented Dataset

In [8]:
import time

print("Starting FULL Run on CP Dataset (Batch=16, 640x640, Epochs=200)...")
t0 = time.time()

results = model.train(
    data='datasets/dataset_yolo_single_class_cp/data.yaml',
    epochs=200,
    patience=30,
    imgsz=640,
    batch=16,
    name='EXP17-YOLO11m-CP-FULL',
    cache=False,
    amp=True,
    exist_ok=True
)

t1 = time.time()
print(f"\nFull Run Complete in {(t1 - t0)/3600:.2f} hours.")
print("Best weights saved to: runs/detect/EXP17-YOLO11m-CP-FULL/weights/best.pt")


Starting FULL Run on CP Dataset (Batch=16, 640x640, Epochs=200)...
Ultralytics 8.4.153 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/dataset_yolo_single_class_cp/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.9